# Cat vs Dog Dataset Exploration

## Objective

This notebook examines the raw dataset before preprocessing or model training.

We will:
- inspect the dataset structure;
- count cat and dog images;
- verify that images can be opened;
- examine image formats and dimensions;
- display representative examples;
- identify corrupted, duplicated, or suspicious images;
- decide what cleaning is required.

In [2]:
# import the libraries
from pathlib import Path 
from collections import Counter

import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

from PIL import Image

Matplotlib is building the font cache; this may take a moment.


In [3]:
# defining the dataset location
DATA_DIR = Path("../data/raw")

print("Dataset directory:", DATA_DIR.resolve())
print("Directory exists:", DATA_DIR.exists())


Dataset directory: /Users/sonya/cat-dog/data/raw
Directory exists: True


In [4]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_paths = [
    path
    for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"Images found: {len(image_paths)}")

Images found: 24998


In [5]:
image_paths[:10]

[PosixPath('../data/raw/Cat/9733.jpg'),
 PosixPath('../data/raw/Cat/63.jpg'),
 PosixPath('../data/raw/Cat/6400.jpg'),
 PosixPath('../data/raw/Cat/823.jpg'),
 PosixPath('../data/raw/Cat/4217.jpg'),
 PosixPath('../data/raw/Cat/3578.jpg'),
 PosixPath('../data/raw/Cat/10321.jpg'),
 PosixPath('../data/raw/Cat/2666.jpg'),
 PosixPath('../data/raw/Cat/5109.jpg'),
 PosixPath('../data/raw/Cat/11981.jpg')]

In [6]:
# Inspecting the folder structure

for item in DATA_DIR.iterdir():
    print(item.name, "->", "Directory" if item.is_dir() else "File")

Cat -> Directory
Dog -> Directory


In [7]:
# Verifying the contents
for class_folder in sorted(DATA_DIR.iterdir()):
    if class_folder.is_dir():
        files = list(class_folder.iterdir())

        print(f"\nFolder: {class_folder.name}")
        print(f"Number of files: {len(files)}")

        for file in files[:5]: 
            print(" ", file.name)


Folder: Cat
Number of files: 12499
  9733.jpg
  63.jpg
  6400.jpg
  823.jpg
  4217.jpg

Folder: Dog
Number of files: 12499
  9733.jpg
  63.jpg
  6400.jpg
  823.jpg
  4217.jpg


In [8]:
# Creating the label function
def get_label(path: Path) -> str:
    folder_name = path.parent.name.lower()

    if folder_name == "cat":
        return "cat"
    elif folder_name == "dog":
        return "dog"
    else:
        return "unknown"

In [12]:
for path in image_paths[:10]:
    print(path.name, "->", get_label(path))

9733.jpg -> cat
63.jpg -> cat
6400.jpg -> cat
823.jpg -> cat
4217.jpg -> cat
3578.jpg -> cat
10321.jpg -> cat
2666.jpg -> cat
5109.jpg -> cat
11981.jpg -> cat


In [13]:
# Checking whether all labels are correct
from collections import Counter

label_counts = Counter(get_label(path) for path in image_paths)
label_counts

Counter({'cat': 12499, 'dog': 12499})

In [14]:
# Creating the image-inspection function
def inspect_image(path: Path) -> dict:
    record = {
        "filepath": str(path),
        "relative_part": str(path.relative_to(DATA_DIR)),
        "filename": path.name,
        "label": get_label(path),
        "width": None,
        "height": None,
        "format": None,
        "mode": None,
        "valid": False,
        "error": None,
    }

    try:
        # Checking whether the file is structurally valid
        with Image.open(path) as image:
            image.verify()
        # Reopening cause verify() makes the previous object unusable
        with Image.open(path) as image:
            image.load()

            record["width"] = image.width
            record["height"] = image.height 
            record["format"] = image.format
            record["mode"] = image.mode
            record["valid"] = True

    except Exception as error:
        record["error"] = f"{type(error).__name__}: {error}"
    return record

In [15]:
test_record = inspect_image(image_paths[0])
test_record

{'filepath': '../data/raw/Cat/9733.jpg',
 'relative_part': 'Cat/9733.jpg',
 'filename': '9733.jpg',
 'label': 'cat',
 'width': 365,
 'height': 500,
 'format': 'JPEG',
 'mode': 'RGB',
 'valid': True,
 'error': None}

In [16]:
records = [inspect_image(path) for path in image_paths]

/Users/sonya/cat-dog/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [17]:
# Creating a DataFrame to store image information
df = pd.DataFrame(records)

print("DataFrame shape:", df.shape)
df.head()

DataFrame shape: (24998, 10)


,filepath,relative_part,filename,label,width,height,format,mode,valid,error
0,../data/raw/Cat/9733.jpg,Cat/9733.jpg,9733.jpg,cat,365,500,JPEG,RGB,True,None
1,../data/raw/Cat/63.jpg,Cat/63.jpg,63.jpg,cat,500,375,JPEG,RGB,True,None
2,../data/raw/Cat/6400.jpg,Cat/6400.jpg,6400.jpg,cat,320,240,JPEG,RGB,True,None
3,../data/raw/Cat/823.jpg,Cat/823.jpg,823.jpg,cat,500,417,JPEG,RGB,True,None
4,../data/raw/Cat/4217.jpg,Cat/4217.jpg,4217.jpg,cat,500,375,JPEG,RGB,True,None


In [18]:
# Checking the column types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24998 entries, 0 to 24997
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   filepath       24998 non-null  str   
 1   relative_part  24998 non-null  str   
 2   filename       24998 non-null  str   
 3   label          24998 non-null  str   
 4   width          24998 non-null  int64 
 5   height         24998 non-null  int64 
 6   format         24998 non-null  str   
 7   mode           24998 non-null  str   
 8   valid          24998 non-null  bool  
 9   error          0 non-null      object
dtypes: bool(1), int64(2), object(1), str(6)
memory usage: 1.7+ MB


In [23]:
# Counting valid and invalid images

print(f"Total images: {len(df):,}")
print(f"Valid images: {df['valid'].sum(),}")
print(f"Invalid images: {(~df['valid']).sum():,}")

Total images: 24,998
Valid images: (np.int64(24998),)
Invalid images: 0


In [33]:
# Creating a separatre DataFrame for invalid files

invalid_images = df.loc[
    ~df["valid"],
    ["filepath", "label", "error"]
]

invalid_images

,filepath,label,error


In [34]:
# Checking whether every label was identified
df["label"].value_counts(dropna=False)

label
cat    12499
dog    12499
Name: count, dtype: int64

In [35]:
df[df["label"] == "unknown"]

,filepath,relative_part,filename,label,width,height,format,mode,valid,error


In [36]:
# Creating a clean working DataFrame 
valid_df = df[
    df["valid"]
    & df["label"].isin(["cat", "dog"])

].copy()

print("Valid working images:", len(valid_df))

Valid working images: 24998


In [37]:
valid_df.head()


,filepath,relative_part,filename,label,width,height,format,mode,valid,error
0,../data/raw/Cat/9733.jpg,Cat/9733.jpg,9733.jpg,cat,365,500,JPEG,RGB,True,None
1,../data/raw/Cat/63.jpg,Cat/63.jpg,63.jpg,cat,500,375,JPEG,RGB,True,None
2,../data/raw/Cat/6400.jpg,Cat/6400.jpg,6400.jpg,cat,320,240,JPEG,RGB,True,None
3,../data/raw/Cat/823.jpg,Cat/823.jpg,823.jpg,cat,500,417,JPEG,RGB,True,None
4,../data/raw/Cat/4217.jpg,Cat/4217.jpg,4217.jpg,cat,500,375,JPEG,RGB,True,None


In [39]:
# Saving the metadata catalogue
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

catalogue_path = REPORTS_DIR / "dataset_catalogue.csv"

df.to_csv(catalogue_path, index=False)

print("Catalogue saved to:")
print(catalogue_path.resolve())

Catalogue saved to:
/Users/sonya/cat-dog/reports/dataset_catalogue.csv
